# Tutorial 8 · Local explanations with LIME

AI in Rehabilitation and Healthcare (376-1229-00L) · HS 2026 · Explainable machine learning I: LIME

In this notebook you will:
1. Load the **activity recognition** feature dataset (`features_df_10s_3s.csv`).
2. Train a **black-box classifier** (Random Forest).
3. Use **LIME (Local Interpretable Model-agnostic Explanations)** to:
   - Explain individual predictions.
   - Compare explanations for **correct vs. incorrect** predictions.
   - Check **stability** of explanations (different seeds).
   - Check **faithfulness** (do important features actually matter for the model?).

> **Goal:** understand what the model uses *locally* around specific windows, and when to trust (or distrust) those explanations.


**Evaluate before you explain.** An explanation describes what the model did; it is worth reading only after the model has been scored on people it never saw, beside a baseline. This notebook does both before the first explanation.

**Data.** `features_df_10s_3s.csv` is computed from the public MotionSense dataset (iPhone accelerometer and gyroscope at 50 Hz, 24 people, 6 activities) with Tutorial 2's recipe: a 10 Hz low-pass filter, then ten features per axis on 10-second windows with a 3-second step. Cite it as: Malekzadeh M, Clegg RG, Cavallaro A, Haddadi H. Mobile sensor data anonymization. *IoTDI '19*, ACM, 2019, pp. 49–58. doi:10.1145/3302505.3310068.

**With the course coach.** Run *T8.00* with your code from SCAI Learning (*Your data → Connect a notebook*). The coach then sees the code of each cell you run, and the *Report* cells send it numbers about your split, your scores and your model, never your data and never a cell's output. Ask it about a cell by its name, for example *"why does T8.05 split by person?"*. Without a code the notebook runs the same and sends nothing.


In [ ]:
# T8.00 Connect the course coach (optional)
# In SCAI Learning, open Your data → Connect a notebook and copy the six-letter code. Paste it
# between the quotes and run this cell. Left empty, nothing is installed and nothing is sent.
# Connected, the coach receives the code of each cell you run and the numbers the Report cells
# compute: never your data, never a cell's output.
COACH_CODE = ""
coach = globals().get("coach")          # connected already with the app's snippet? keep it
if COACH_CODE and coach is None:
    %pip install -q https://app.scai-learning.ch/companion.tar.gz
    import rehab_coach as coach
    coach.connect(COACH_CODE)
    coach.watch()
print("Coach connected: the cells you run from here on reach it." if coach else "Working without the coach.")


## 1. Setup and load the dataset


In [ ]:
# T8.01 Install lime
%pip install -q lime


In [ ]:
# T8.02 Mount Google Drive
# Colab only. On Jupyter on your own computer, skip this cell and set data_path below to your folder.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# T8.03 Load the feature table
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

RANDOM_STATE = 42

# --- Load the featured dataset ---
data_path = Path("/content/drive/MyDrive/Colab Data/ai_in_rehab_2026_tutorial/")   # Jupyter on your own computer: the folder that holds the file
load_path = data_path / "features_df_10s_3s.csv"


print(f"Loading data from: {load_path}")
features_df = pd.read_csv(load_path)
print("Shape:", features_df.shape)
display(features_df.head())


## 2. Prepare data for classification


In [ ]:
# T8.04 Labels, features and people
# Encode labels (activity) into numeric values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(features_df["activity"])

# Extract features (drop metadata columns)
meta_cols = ["activity", "subject", "trial", "window"]
feature_cols = [c for c in features_df.columns if c not in meta_cols]

X = features_df[feature_cols].to_numpy().astype(float)
subjects = features_df["subject"].to_numpy()

print("Number of classes:", len(label_encoder.classes_))
print("Classes:", label_encoder.classes_[:10], "...")
print("Number of features:", len(feature_cols))


### 2.1 Subject-wise train–test split


In [ ]:
# T8.05 Split by person
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(splitter.split(X, y, groups=subjects))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

subjects_train = subjects[train_idx]
subjects_test = subjects[test_idx]

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Unique subjects in train:", np.unique(subjects_train))
print("Unique subjects in test:", np.unique(subjects_test))
print("People on both sides:", len(set(subjects_train) & set(subjects_test)))


In [ ]:
# T8.06 Report the split and the baseline
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T8.00).
if coach:
    coach.baseline(y_train)                     # the zero-rate score: always the commonest activity
    coach.report((train_idx, test_idx), subjects, y, tutorial="T8")   # opens the coach's explanation gate if the split is clean
else:
    print("Not connected: nothing sent. Run T8.00 with your code to use the coach.")


### 2.2 (Optional) Standardize features for distance-based LIME


In [ ]:
# T8.07 Standardise on the training people
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Scaled train shape:", X_train_s.shape)


## 3. Train a black-box model (Random Forest)


In [ ]:
# T8.08 Train the black box and score it
from sklearn.metrics import balanced_accuracy_score

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

model.fit(X_train_s, y_train)

proba_test = model.predict_proba(X_test_s)
pred_test  = model.predict(X_test_s)

acc = accuracy_score(y_test, pred_test)
f1  = f1_score(y_test, pred_test, average="macro")

bal = balanced_accuracy_score(y_test, pred_test)
values, counts = np.unique(y_train, return_counts=True)
zero_rate = (y_test == values[counts.argmax()]).mean()
print(f"Random Forest — Accuracy: {acc:.3f}, balanced accuracy: {bal:.3f}, F1-macro: {f1:.3f}")
print(f"Zero-rate (always '{label_encoder.classes_[values[counts.argmax()]]}') accuracy: {zero_rate:.3f}")
print("\nClassification report:")
print(classification_report(y_test, pred_test, target_names=label_encoder.classes_))

# Simple confusion matrix
cm = confusion_matrix(y_test, pred_test)
plt.figure(figsize=(6,4))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# T8.09 Report the model's scores
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T8.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, model=model, tutorial="T8",
                 scores={"accuracy": acc, "balanced_accuracy": bal, "f1_macro": f1})
else:
    print("Not connected: nothing sent. Run T8.00 with your code to use the coach.")


## 4. LIME: Local Interpretable Model-agnostic Explanations


**LIME idea (very briefly):**  
Around a single instance \( x_0 \), we:
1. Generate many **perturbed samples** near \( x_0 \).
2. Ask the black-box model for their predictions.
3. Fit a **simple, local surrogate** (usually a sparse linear model) on those samples, weighted by distance to \( x_0 \).
4. Use the surrogate coefficients as **local feature importances**.

We will use `lime.lime_tabular.LimeTabularExplainer` on our scaled features.

Two numbers come with every explanation. `exp.score` is the **R² of the surrogate**: how well the straight line fits the forest's answers around this window. `exp.local_pred` is what the surrogate predicts for the window itself, to compare with the forest's probability. A list of weights from a surrogate that fits badly describes the surrogate, not the model.


In [ ]:
# T8.10 Install the explainer
from lime import lime_tabular

class_names = list(label_encoder.classes_)

explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train_s,
    feature_names=feature_cols,
    class_names=class_names,
    discretize_continuous=True,
    mode="classification"
)

explainer


explained = set()          # which test windows have been explained so far
surrogate_fits = []        # the R² of each explanation's surrogate


### 4.1 Explain a single test prediction


In [ ]:
# T8.11 Explain one test window
# Pick a random test index
np.random.seed(RANDOM_STATE)
i = np.random.randint(0, X_test_s.shape[0])

x0 = X_test_s[i]
true_label_idx = y_test[i]
pred_label_idx = pred_test[i]

print("Test index:", i)
print("True activity    :", label_encoder.inverse_transform([true_label_idx])[0])
print("Predicted activity:", label_encoder.inverse_transform([pred_label_idx])[0])

# Explain this instance
exp = explainer.explain_instance(
    data_row=x0,
    predict_fn=model.predict_proba,
    num_features=10,          # top features to show
    top_labels=1              # focus on predicted class
)

explained.add(int(i)); surrogate_fits.append(exp.score)
print(f"Surrogate R²: {exp.score:.2f} · surrogate predicts {exp.local_pred[0]:.2f}, the forest {model.predict_proba(x0.reshape(1, -1))[0, pred_label_idx]:.2f}")

# Show as list (text)
exp_list = exp.as_list(label=pred_label_idx)
print("\nLIME explanation (feature, weight) for predicted class:")
for feat, weight in exp_list:
    print(f"{feat:40s}  weight={weight:+.4f}")

# Show as a matplotlib figure
fig = exp.as_pyplot_figure(label=pred_label_idx)
plt.tight_layout()
plt.show()


In [ ]:
# T8.12 Show the explanation as a table
# exp.show_in_notebook() imports a name IPython 9 removed; this shows the same page on every version.
from IPython.display import HTML
display(HTML(exp.as_html(show_table=True, show_all=True)))


### 4.2 Compare LIME explanations for correct vs incorrect predictions


In [ ]:
# T8.13 Correct against incorrect
# Find one correctly classified and one misclassified example
correct_idx = np.where(pred_test == y_test)[0]
wrong_idx   = np.where(pred_test != y_test)[0]

print("Number correct:", len(correct_idx), " Number wrong:", len(wrong_idx))

if len(correct_idx) > 0 and len(wrong_idx) > 0:
    i_correct = correct_idx[0]
    i_wrong   = wrong_idx[0]
else:
    i_correct = 0
    i_wrong   = 1

def explain_index(idx, title=""):
    x0 = X_test_s[idx]
    true_idx = y_test[idx]
    pred_idx = pred_test[idx]
    print("="*60)
    print(title)
    print("Test index:", idx)
    print("True activity    :", label_encoder.inverse_transform([true_idx])[0])
    print("Predicted activity:", label_encoder.inverse_transform([pred_idx])[0])

    exp = explainer.explain_instance(
        data_row=x0,
        predict_fn=model.predict_proba,
        num_features=10,
        top_labels=1
    )
    explained.add(int(idx)); surrogate_fits.append(exp.score)
    print(f"Surrogate R²: {exp.score:.2f}")
    fig = exp.as_pyplot_figure(label=pred_idx)
    plt.tight_layout()
    plt.show()

explain_index(i_correct, title="CORRECT example")
explain_index(i_wrong,   title="WRONG example")


### 4.3 Stability: Does LIME give similar explanations with different seeds?


In [ ]:
# T8.14 Stability across seeds
idx = i_correct  # use the correct example for stability check
x0  = X_test_s[idx]
pred_idx = pred_test[idx]

def lime_weights_for_seed(seed, num_features=10):
    np.random.seed(seed)
    exp = explainer.explain_instance(
        data_row=x0,
        predict_fn=model.predict_proba,
        num_features=num_features,
        top_labels=1
    )
    # return as dict {feature: weight}
    surrogate_fits.append(exp.score)
    w = dict(exp.as_list(label=pred_idx))
    return w

seeds = [0, 1, 2, 3, 4]
weights_list = [lime_weights_for_seed(s) for s in seeds]

# Combine into one DataFrame
all_features = set().union(*[w.keys() for w in weights_list])
rows = []
for feat in sorted(all_features):
    row = {"feature": feat}
    for s, w in zip(seeds, weights_list):
        row[f"seed_{s}"] = w.get(feat, 0.0)
    rows.append(row)

stability_df = pd.DataFrame(rows)
display(stability_df)

# Simple visualization: standard deviation of weights across seeds
stability_df["std_weight"] = stability_df[[f"seed_{s}" for s in seeds]].std(axis=1)
stability_df_sorted = stability_df.sort_values("std_weight", ascending=False)
print("Features with most unstable LIME weights (on this instance):")
display(stability_df_sorted.head(10))

# How many of seed 0's top five features drop out of the top five under another seed?
top5 = [set(sorted(w, key=lambda f: -abs(w[f]))[:5]) for w in weights_list]
churn = len(top5[0] - set.intersection(*top5))
print(f"{churn} of seed 0's top five features are not in the top five for every seed")


### 4.4 Faithfulness: do top LIME features actually matter?
*What "removing" means here.* Setting a standardised feature to 0 sets it to the training mean. Each feature is found by its column index (`exp.as_map()`), not by parsing LIME's text, so `-0.43 < acc_x_std <= 0.12` cannot be mistaken for another column. The grey line removes the same number of **random** features: LIME's features should matter more than those.


In [ ]:
# T8.15 Faithfulness
idx = i_correct
x0_orig = X_test_s[idx:idx+1]  # keep 2D
pred_idx = int(pred_test[idx])

exp = explainer.explain_instance(
    data_row=x0_orig[0],
    predict_fn=model.predict_proba,
    num_features=10,
    top_labels=1
)
surrogate_fits.append(exp.score)
ranked = sorted(exp.as_map()[pred_idx], key=lambda t: -abs(t[1]))    # (column index, weight), largest first
print("Top 10 LIME features and weights:")
for j, w in ranked:
    print(f"{feature_cols[j]:40s}  {w:+.4f}")

base_proba = model.predict_proba(x0_orig)[0, pred_idx]
ks = list(range(0, 11))
def drop_after_zeroing(cols):
    x_mod = x0_orig.copy(); x_mod[0, cols] = 0.0          # 0 in standardised space: the training mean
    return base_proba - model.predict_proba(x_mod)[0, pred_idx]

delta = [drop_after_zeroing([j for j, _ in ranked[:k]]) for k in ks]
r = np.random.default_rng(0)
delta_random = [np.mean([drop_after_zeroing(r.choice(len(feature_cols), k, replace=False)) for _ in range(20)]) for k in ks]

plt.figure()
plt.plot(ks, delta, marker="o", label="top LIME features")
plt.plot(ks, delta_random, marker=".", color="grey", label="random features (mean of 20)")
plt.xlabel("Number of features set to the training mean")
plt.ylabel(f"Decrease in P(class={label_encoder.inverse_transform([pred_idx])[0]})")
plt.title("Faithfulness check: removing top LIME features")
plt.legend(); plt.grid(True)
plt.show()


In [ ]:
# T8.16 Report the explanations
# Numbers about the cells above, computed here and sent to the course coach (nothing else is
# sent, and nothing at all without T8.00).
if coach:
    coach.report((train_idx, test_idx), subjects, y, tutorial="T8",
                 lesson={"explained_cases": len(explained), "surrogate_r2": float(np.median(surrogate_fits)),
                         "top_feature_churn": churn})
else:
    print("Not connected: nothing sent. Run T8.00 with your code to use the coach.")


## 5. ✍️ Exercise

Pick **one activity class** (e.g., "walking") and **one different model** (e.g., MLP):

1. **Collect multiple explanations**
   - Find **5 test samples** predicted as that class (at least 2 should be wrong, if possible).
   - Generate LIME explanations for each of them (same settings as above).

2. **Compare patterns**
   - Which sensor features appear most often across explanations?
   - Are there clear differences between **correct** and **incorrect** predictions?

> OPTIONAL: Try changing `num_features`, `discretize_continuous`, or the kernel width in `LimeTabularExplainer` and see how much the explanation changes.

For each of your explanations, write down the surrogate's R². If you are connected, ask the coach to review your comparison: it will ask how the five windows were chosen and whether the pattern survives another seed.
